In [ ]:
!pip install 'crewai[tools]' -q

In [ ]:
# Import libraries
import os
import re

from crewai import Agent, Crew, Process, Task, LLM
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
from crewai.flow.flow import Flow, start, listen, router, or_

from pydantic import BaseModel, Field
from typing import List, Dict, Literal

from IPython.display import display, Markdown

In [ ]:
# Retrieve API keys and store them as environment variables

from google.colab import userdata

gemini_api_key = userdata.get("GEMINI_API_KEY")
serper_api_key = userdata.get("SERPER_API_KEY")

if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY not found in Colab Secrets.")

if not serper_api_key:
    raise ValueError("SERPER_API_KEY not found in Colab Secrets.")

os.environ["GEMINI_API_KEY"] = gemini_api_key
os.environ["SERPER_API_KEY"] = serper_api_key

In [ ]:
# Define the LLM
llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    temperature=0.5
)

# User Inputs

In [ ]:
# Collect interview details
interviewer = input("Enter the name of the interviewer (leave blank if unknown): ").strip()
company = input("Enter the name of the company: ").strip()
job_position = input("Enter the job position: ").strip()
job_description = input("Enter the job description: ").strip()

if not interviewer:
    interviewer = None

# Agents

In [ ]:
research_agent = Agent(
    role="Technical Interview Researcher",
    goal="""
    Research the target company, job role, required technologies, likely interview
    topics, and the professional background of the interviewer in order to identify
    the most relevant areas the candidate should prepare for.
    """,
    backstory="""
    You are an expert technical interview researcher with experience in software
    engineering recruitment and hiring processes.

    You investigate the company, job description, engineering stack, products,
    recent technical developments, and publicly reported interview experiences.

    You also research the interviewer's publicly available professional background,
    including their current role, previous roles, technical expertise, projects,
    conference talks, publications, GitHub activity, and areas of specialization.

    You do not speculate about the interviewer personally. Instead, you identify
    professional topics they may be especially knowledgeable about and therefore
    more likely to discuss during an interview.

    You combine these signals into evidence-based preparation topics and clearly
    distinguish verified information, strong signals, and uncertain assumptions.
    """,
    llm=llm,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True
)

In [ ]:
coach_agent = Agent(
    role="Interview Coach",
    goal=f"""
    Help the candidate prepare effectively for the {job_position} interview at {company}
    by evaluating their answers, identifying strengths and weaknesses, and giving
    clear, actionable feedback that improves both content and delivery.
    """,
    backstory=f"""
    You are an experienced interview coach who helps candidates prepare for
    technical and behavioral interviews.

    You understand how interviewers evaluate answers based on relevance, clarity,
    structure, technical depth, communication, and evidence from past experience.

    You adapt your feedback to the specific role of {job_position} at {company},
    using the job requirements and interview context when available.

    Your feedback is constructive, specific, and practical. You explain what was
    done well, what could be improved, and how the candidate could give a stronger
    answer next time.

    You do not invent missing experience or facts about the candidate. When an
    answer lacks detail, you point out what is missing rather than making assumptions.
    """,
    llm=llm,
    verbose=True
)

# Tasks

In [ ]:
# Define the task for researching the company
research_company_task = Task(
    description=f"""
    Research {company} with the goal of helping a candidate prepare for an interview.

    Focus on:
    - what the company does, its products or services, and target customers
    - its position in the market and main competitors
    - recent company news, product launches, partnerships, funding, expansion,
      leadership changes, or strategic developments from the last 12 months
    - important trends affecting the company's industry
    - the company's culture, values, and ways of working when reliable information is available
    - any signals that are especially relevant to the {job_position} role

    Prioritize recent and credible public sources.
    Clearly distinguish verified facts from reasonable interpretation.
    Do not invent information when evidence is unavailable.
    """,
    expected_output=f"""
    A concise interview-preparation report in Markdown about {company}, including:

    - Company overview
    - Products/services and business model
    - Market position and competitors
    - Important developments from the last 12 months
    - Relevant industry trends
    - Culture and working-style signals
    - Key insights relevant to the {job_position} interview
    - 5-8 interview preparation takeaways for the candidate

    Keep the report evidence-based, practical, and focused on information that
    could realistically help in an interview.
    """,
    agent=research_agent
)

In [ ]:
# Define the task for researching the interviewer
research_person_task = Task(
    description=f"""
    Research the publicly available professional background of {interviewer}
    in the context of their role at {company}.

    Focus on:
    - current role and responsibilities
    - previous relevant roles
    - technical or professional areas of expertise
    - notable projects, products, or initiatives they have worked on
    - conference talks, publications, GitHub activity, interviews, or other
      professional content when publicly available
    - topics they appear to be especially knowledgeable about
    - connections between their background and the {job_position} role

    Verify that the information refers to the correct person before drawing conclusions.
    Use company, role, location, or other professional context to disambiguate people
    with the same or similar names.

    Do not speculate about personal characteristics, private life, opinions, or behavior.
    Clearly distinguish verified facts, strong professional signals, and uncertain information.
    If the correct person cannot be identified confidently, state that explicitly.
    """,
    expected_output=f"""
    A concise Markdown interview-preparation report about {interviewer}, including:

    - Current professional role
    - Relevant career background
    - Main areas of expertise
    - Notable professional projects, talks, publications, or public activity
    - Professional topics they may be especially familiar with
    - How their background may be relevant to the {job_position} interview
    - 3-5 preparation topics the candidate should review

    Include uncertainty where appropriate and avoid unsupported assumptions.
    """,
    agent=research_agent
)

In [ ]:
# Define the structure of the Interview Questions
class InterviewQuestion(BaseModel):
    category: str
    question: str


class InterviewQuestionList(BaseModel):
    questions: List[InterviewQuestion]

In [ ]:
# Build preparation tasks dynamically
preparation_tasks = [research_company_task]
question_context = [research_company_task]

if interviewer:
    preparation_tasks.append(research_person_task)
    question_context.append(research_person_task)

# Define the question-generation task
define_questions_task = Task(
    description=f"""
    Create a realistic set of interview questions for the {job_position}
    position at {company}.

    Use:
    - the job description provided below
    - the company research
    - interviewer research, if available

    Job description:
    {job_description}

    Prioritize questions that directly test the skills, responsibilities,
    and competencies described in the job description.

    Use the company research to make questions relevant to the company's
    business, culture, products, and working environment.

    If interviewer research is available, use it only when there is a clear
    professional connection between the interviewer's background and a likely
    interview topic.

    Avoid duplicate, overly generic, or unsupported questions.
    """,
    expected_output="""
    Exactly 20 interview questions in Markdown format, grouped into:

    ## Culture and Team Fit
    5 questions

    ## Job Position Fit
    7 questions

    ## Background and Ways of Working
    5 questions

    ## Growth Mindset
    3 questions

    Do not include answers or explanations.
    """,
    agent=coach_agent,
    context=question_context,
    output_pydantic=InterviewQuestionList
)

preparation_tasks.append(define_questions_task)

# The Crew

In [ ]:
# Create and run the Preparation Crew
preparation_crew = Crew(
    agents=[research_agent, coach_agent],
    tasks=preparation_tasks,
    process=Process.sequential,
    verbose=True
)

In [ ]:
# Run the Preparation Crew
result = await preparation_crew.kickoff_async()

display(Markdown(result.raw))

In [ ]:
# Check the Crew outcome

# Company Research report
company_report = result.tasks_output[0].raw

display(Markdown("## Company Research"))
display(Markdown(company_report))


# Interviewer Research report, if available
if interviewer:
    interviewer_report = result.tasks_output[1].raw

    display(Markdown("## Interviewer Research"))
    display(Markdown(interviewer_report))
else:
    interviewer_report = None


# Structured interview questions
questions_output = result.tasks_output[-1].pydantic

interview_questions = [
    item.question
    for item in questions_output.questions
]


# Display the questions nicely
display(Markdown("## Interview Questions"))

for i, item in enumerate(questions_output.questions, start=1):
    print(f"{i}. [{item.category}] {item.question}")

# Interview Preparation

In [ ]:
# Define the feedback crew
def build_feedback_crew():
    feedback_task = Task(
        description=(
            "The candidate is practicing for a job interview.\n\n"
            f"Job position:\n{job_position}\n\n"
            "Interview question:\n"
            "{current_question}\n\n"
            "Candidate answer:\n"
            "{user_answer}\n\n"
            "Evaluate the answer as an interview coach.\n"
            "Focus on relevance, clarity, structure, specificity, evidence, "
            "technical or role-related depth when applicable, and communication.\n\n"
            "Do not invent experience or details that the candidate did not mention. "
            "If important information is missing, explain what should be added."
        ),
        expected_output=(
            "Return concise Markdown with exactly these sections:\n\n"
            "## Score\n"
            "Give a score from 1 to 10 and one short justification.\n\n"
            "## What was good\n"
            "Identify the strongest parts of the answer.\n\n"
            "## What needs improvement\n"
            "Explain what is unclear, weak, missing, or too generic.\n\n"
            "## How to take it to the next level\n"
            "Give 2-3 specific actions for improving the answer."
        ),
        agent=coach_agent
    )

    return Crew(
        agents=[coach_agent],
        tasks=[feedback_task],
        process=Process.sequential,
        verbose=True,
        memory=False,
        planning=False
    )

In [ ]:
# Define the summary crew
def build_summary_crew():
    summary_task = Task(
        description=(
            "The candidate completed a mock interview.\n\n"
            f"Company:\n{company}\n\n"
            f"Job position:\n{job_position}\n\n"
            "Interview history:\n"
            "{history_text}\n\n"
            "Create a final assessment of the candidate's interview performance.\n"
            "Base the assessment only on the questions, answers, and feedback in the history.\n"
            "Look for repeated patterns across answers rather than judging isolated mistakes.\n"
            "Do not invent skills, weaknesses, or experiences that are not supported by the history."
        ),
        expected_output=(
            "Return concise Markdown with exactly these sections:\n\n"
            "## Overall strengths\n"
            "Summarize the strongest recurring aspects of the candidate's answers.\n\n"
            "## Main weaknesses\n"
            "Summarize the most important recurring weaknesses or gaps.\n\n"
            "## Top 3 priorities before the real interview\n"
            "Give exactly 3 specific and actionable preparation priorities.\n\n"
            "## Final readiness score\n"
            "Give a score from 1 to 10 and briefly explain the rating."
        ),
        agent=coach_agent
    )

    return Crew(
        agents=[coach_agent],
        tasks=[summary_task],
        process=Process.sequential,
        verbose=True,
        memory=False,
        planning=False
    )

In [ ]:
# Define the Flow state
class InterviewHistoryItem(BaseModel):
    question_number: int
    question: str
    answer: str
    feedback: str


class InterviewState(BaseModel):
    interview_questions: List[str] = Field(default_factory=list)
    current_index: int = 0
    current_question: str = ""
    user_answer: str = ""
    feedback: str = ""
    history: List[InterviewHistoryItem] = Field(default_factory=list)
    final_summary: str = ""

In [ ]:
# Build the interview prep flow

class InterviewPrepFlow(Flow[InterviewState]):

    @start()
    async def run_interview(self):
        print("Starting mock interview...")

        if not self.state.interview_questions:
            raise ValueError("No interview questions found.")

        # Workshop demo: use only the first 3 questions
        questions = self.state.interview_questions[:3]

        self.state.current_index = 0
        self.state.history = []

        total_questions = len(questions)

        while self.state.current_index < total_questions:
            question = questions[self.state.current_index]
            self.state.current_question = question

            print(f"\n{'=' * 70}")
            print(f"Question {self.state.current_index + 1}/{total_questions}")
            print(f"{'=' * 70}")
            print(f"\n[{company} Interviewer] {question}")

            # Collect answer
            answer = input("\nYour answer: ").strip()

            while not answer:
                answer = input("Please enter an answer: ").strip()

            self.state.user_answer = answer

            # Run the Feedback Crew
            feedback_crew = build_feedback_crew()

            feedback_result = await feedback_crew.kickoff_async(
                inputs={
                    "current_question": self.state.current_question,
                    "user_answer": self.state.user_answer
                }
            )

            self.state.feedback = feedback_result.raw

            # Display feedback
            print(f"\n{'-' * 70}")
            print("Coach feedback")
            print(f"{'-' * 70}")
            display(Markdown(self.state.feedback))

            # Store this interview round in the Flow state
            self.state.history.append(
                InterviewHistoryItem(
                    question_number=self.state.current_index + 1,
                    question=self.state.current_question,
                    answer=self.state.user_answer,
                    feedback=self.state.feedback
                )
            )

            self.state.current_index += 1

        print(f"\n{'=' * 70}")
        print("Mock interview completed")
        print(f"You answered {len(self.state.history)} questions.")
        print(f"{'=' * 70}")

        # Prepare interview history for the Summary Crew
        history_text = "\n\n".join(
            [
                f"Question {item.question_number}:\n"
                f"{item.question}\n\n"
                f"Answer:\n"
                f"{item.answer}\n\n"
                f"Feedback:\n"
                f"{item.feedback}"
                for item in self.state.history
            ]
        )

        # Run the Summary Crew
        summary_crew = build_summary_crew()

        final_summary_result = await summary_crew.kickoff_async(
            inputs={
                "history_text": history_text
            }
        )

        self.state.final_summary = final_summary_result.raw

        # Display final summary
        print(f"\n{'=' * 70}")
        print("FINAL SUMMARY")
        print(f"{'=' * 70}")
        display(Markdown(self.state.final_summary))

        return {
            "history": self.state.history,
            "final_summary": self.state.final_summary
        }

In [ ]:
# Start the flow

flow = InterviewPrepFlow()
flow.state.interview_questions = interview_questions

# Optional: visualize the flow
# flow.plot("interview_prep_flow")

result = await flow.kickoff_async()
display(Markdown("## Final Assessment"))
display(Markdown(result["final_summary"]))